## 04_GNN 성능평가 — best_model.pt 로드 후 실행

03_GNN_모델링에서 학습이 완료된 `best_model.pt`를 로드한 뒤, **학습 없이** 다음만 수행합니다:
- 최종 성능 평가 (Recall@K, NDCG@K) 및 인기도 베이스라인 비교
- GNN 유저 임베딩 저장 및 KMeans 군집
- 군집 결과·메트릭·대시보드용 CSV 저장

**실행 전 확인**: `../Data Folder/best_model.pt` 파일이 있어야 합니다.


In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder
import os

CLIP_DIM = 512  # CLIP ViT-B/32 출력 차원 (03_GNN_모델링과 동일)

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
_font_path = '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
if os.path.exists(_font_path):
    fm.fontManager.addfont(_font_path)
    plt.rcParams['font.family'] = fm.FontProperties(fname=_font_path).get_name()
elif os.name == 'nt':
    plt.rc('font', family='Malgun Gothic')
else:
    plt.rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False

# ── 1. 데이터 로드 (03과 동일) ────────────────────────────────────
customers = pd.read_parquet('../Data Folder/H&M dataset/H&m parquet dataset/customers.parquet')
articles  = pd.read_parquet('../Data Folder/H&M dataset/H&m parquet dataset/articles.parquet')
transactions_all = pd.read_parquet('../Data Folder/H&M dataset/H&m parquet dataset/transactions.parquet')
transactions_all['t_dat'] = pd.to_datetime(transactions_all['t_dat'])
# article_id 타입 통일 (merge 에러 방지)
for col in ['article_id']:
    if col in transactions_all.columns:
        transactions_all[col] = transactions_all[col].astype(str)
if 'article_id' in articles.columns:
    articles['article_id'] = articles['article_id'].astype(str)

cutoff_date = pd.Timestamp('2019-12-17')
train_df = transactions_all[transactions_all['t_dat'] <= cutoff_date].copy()
test_end = pd.Timestamp('2019-12-31')
test_df  = transactions_all[(transactions_all['t_dat'] > cutoff_date) & (transactions_all['t_dat'] <= test_end)].copy()

test_users_in_train = set(train_df['customer_id'].unique())
test_df = test_df[test_df['customer_id'].isin(test_users_in_train)].copy()
test_ground_truth = test_df.groupby('customer_id')['article_id'].apply(lambda x: set(x.astype(str))).to_dict()

transactions = train_df
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()
transactions['user_idx'] = user_encoder.fit_transform(transactions['customer_id'])
transactions['item_idx'] = item_encoder.fit_transform(transactions['article_id'])

user_features_df = customers.set_index('customer_id').reindex(user_encoder.classes_)
user_features_df = user_features_df.select_dtypes(include=[np.number]).fillna(0)
user_dict = {idx: torch.tensor(row, dtype=torch.float32) for idx, row in enumerate(user_features_df.values)}

item_features_df = articles.set_index('article_id').reindex(item_encoder.classes_)
item_features_df = item_features_df.select_dtypes(include=[np.number]).fillna(0)

try:
    image_embeddings = np.load('../Data Folder/clip_image_embeddings.npy', allow_pickle=True).item()
    print(f'CLIP 이미지 임베딩 로드 완료 | 상품 수: {len(image_embeddings):,}')
except FileNotFoundError:
    image_embeddings = {}

item_dict = {}
for idx, article_id in enumerate(item_encoder.classes_):
    meta_feature = torch.tensor(item_features_df.iloc[idx].values, dtype=torch.float32)
    try:
        article_id_int = int(article_id)
    except (ValueError, TypeError):
        article_id_int = int(float(article_id))
    img_feature = (torch.tensor(image_embeddings[article_id_int], dtype=torch.float32)
                   if article_id_int in image_embeddings else torch.zeros(CLIP_DIM, dtype=torch.float32))
    item_dict[idx] = torch.cat([meta_feature, img_feature])

print(f'데이터 구성 완료 | 유저: {len(user_encoder.classes_):,} | 아이템: {len(item_encoder.classes_):,} | 거래: {len(transactions):,}')

CLIP 이미지 임베딩 로드 완료 | 상품 수: 75,000
데이터 구성 완료 | 유저: 1,102,708 | 아이템: 78,281 | 거래: 19,792,031


In [2]:
cutoff_date = pd.Timestamp('2019-12-17')
train_df = transactions_all[transactions_all['t_dat'] <= cutoff_date].copy()
transactions = train_df

user_encoder = LabelEncoder()
item_encoder = LabelEncoder()
transactions['user_idx'] = user_encoder.fit_transform(transactions['customer_id'])
transactions['item_idx'] = item_encoder.fit_transform(transactions['article_id'])

print(f"유저: {len(user_encoder.classes_):,} | 아이템: {len(item_encoder.classes_):,}")

유저: 1,102,708 | 아이템: 78,281


In [3]:
import torch.nn as nn
from torch_geometric.nn import LGConv

class MultiModalLightGCN(nn.Module):
    def __init__(self, num_users, num_items, clip_emb_dim=512, embedding_dim=64, num_layers=3):
        super().__init__()
        self.num_users, self.num_items, self.num_layers = num_users, num_items, num_layers
        self.user_emb = nn.Embedding(num_users, embedding_dim)
        self.item_emb = nn.Embedding(num_items, embedding_dim)
        self.clip_transform = nn.Sequential(nn.Linear(clip_emb_dim, 128), nn.GELU(), nn.Linear(128, embedding_dim))
        self.convs = nn.ModuleList([LGConv() for _ in range(num_layers)])
    def forward(self, edge_index, edge_weight, clip_features):
        item_visual_emb = self.clip_transform(clip_features)
        combined_item_emb = self.item_emb.weight + item_visual_emb
        x = torch.cat([self.user_emb.weight, combined_item_emb], dim=0)
        embs = [x]
        for conv in self.convs:
            x = conv(x, edge_index, edge_weight=edge_weight)
            embs.append(x)
        out = torch.mean(torch.stack(embs, dim=0), dim=0)
        return torch.split(out, [self.num_users, self.num_items])
    def predict(self, user_idx, users_out, items_out):
        return torch.matmul(users_out[user_idx], items_out.T)

# item_dict 생성 — CLIP 임베딩 로드
clip_raw = np.load('../Data Folder/clip_image_embeddings.npy', allow_pickle=True).item()

item_dict = {}
for i, aid in enumerate(item_encoder.classes_):
    aid_int = int(aid)
    if aid_int in clip_raw:
        clip_vec = torch.tensor(clip_raw[aid_int], dtype=torch.float32)
    else:
        clip_vec = torch.zeros(CLIP_DIM, dtype=torch.float32)
    item_dict[i] = clip_vec
    
print(f"item_dict 생성 완료: {len(item_dict):,}개")

c:\Users\urina\OneDrive\Desktop\취업 포트폴리오\4) 최종 프로젝트\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


item_dict 생성 완료: 78,281개


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_users = len(user_encoder.classes_)
num_items = len(item_encoder.classes_)

user_indices = torch.tensor(transactions['user_idx'].values, dtype=torch.long)
item_indices = torch.tensor(transactions['item_idx'].values, dtype=torch.long) + num_users
edge_index = torch.stack([user_indices, item_indices], dim=0).to(device)
latest_date = transactions['t_dat'].max()
days_diff = (latest_date - transactions['t_dat']).dt.days
edge_weight = torch.tensor(np.exp(-days_diff.values / 30), dtype=torch.float).to(device)

model = MultiModalLightGCN(num_users=num_users, num_items=num_items, clip_emb_dim=CLIP_DIM, embedding_dim=64, num_layers=3).to(device)
model.load_state_dict(torch.load('../Data Folder/best_model.pt', map_location=device))
print('best_model.pt 로드 완료')


best_model.pt 로드 완료


In [10]:
EVAL_K = 10

def evaluate_popularity_baseline(train_df, item_encoder, test_ground_truth, user_encoder, K=10, sample_users=500):
    """인기도 베이스라인: 학습 데이터 상위 K개 인기 상품으로 동일 유저 평가"""
    top_k_items = train_df['article_id'].value_counts().head(K).index.tolist()
    top_k_set = set(str(x) for x in top_k_items)
    eval_customers = [c for c in list(test_ground_truth.keys()) if c in set(user_encoder.classes_)]
    np.random.seed(42)
    eval_customers = np.random.choice(eval_customers, min(sample_users, len(eval_customers)), replace=False)
    recall_list, ndcg_list = [], []
    for customer_id in eval_customers:
        true_items = test_ground_truth[customer_id]
        hits = top_k_set & true_items
        recall_list.append(len(hits) / min(len(true_items), K))
        dcg = sum(1.0 / np.log2(rank + 2) for rank, iid in enumerate(top_k_items) if str(iid) in true_items)
        idcg = sum(1.0 / np.log2(r + 2) for r in range(min(len(true_items), K)))
        ndcg_list.append(dcg / idcg if idcg > 0 else 0.0)
    return np.mean(recall_list), np.mean(ndcg_list)

def evaluate_model(model, edge_index, edge_weight, item_dict, user_encoder, item_encoder, test_ground_truth, device, K=10, sample_users=500):
    model.eval()
    clip_features = torch.stack([item_dict[i][-CLIP_DIM:] for i in range(num_items)]).to(device)
    
    eval_customers = [c for c in list(test_ground_truth.keys()) if c in set(user_encoder.classes_)]
    np.random.seed(42)
    eval_customers = np.random.choice(eval_customers, min(sample_users, len(eval_customers)), replace=False)
    
    with torch.no_grad():
        users_out, items_out = model(edge_index, edge_weight, clip_features)
        
        # ★ 전체 말고 샘플 유저 인덱스만 추출해서 행렬곱
        sample_indices = [user_encoder.transform([c])[0] for c in eval_customers]
        sample_users_out = users_out[sample_indices]  # (샘플수, 64)
        all_scores = torch.matmul(sample_users_out, items_out.T).cpu().numpy()  # (샘플수, 아이템수)
    
    recall_list, ndcg_list = [], []
    for i, customer_id in enumerate(eval_customers):
        true_items = test_ground_truth[customer_id]
        scores = all_scores[i]  # 이미 계산된 거 슬라이싱
        top_k = np.argsort(scores)[::-1][:K]
        top_k_ids = set(item_encoder.inverse_transform(top_k).astype(str))
        hits = top_k_ids & true_items
        recall_list.append(len(hits) / min(len(true_items), K))
        dcg = sum(1.0/np.log2(rank+2) for rank, iid in enumerate(item_encoder.inverse_transform(top_k)) if str(iid) in true_items)
        idcg = sum(1.0/np.log2(r+2) for r in range(min(len(true_items), K)))
        ndcg_list.append(dcg/idcg if idcg > 0 else 0.0)
    
    return np.mean(recall_list), np.mean(ndcg_list)

In [12]:
# 실제 평가 실행 (best_model.pt 로드 후 추론 — 평가 시간 단축을 위해 500명 샘플링)
pop_recall, pop_ndcg = evaluate_popularity_baseline(train_df, item_encoder, test_ground_truth, user_encoder, K=EVAL_K)
gnn_recall, gnn_ndcg = evaluate_model(model, edge_index, edge_weight, item_dict, user_encoder, item_encoder, test_ground_truth, device, K=EVAL_K)

print(f"GNN  → Recall@{EVAL_K}: {gnn_recall:.4f} | NDCG@{EVAL_K}: {gnn_ndcg:.4f}")
print(f"POP  → Recall@{EVAL_K}: {pop_recall:.4f} | NDCG@{EVAL_K}: {pop_ndcg:.4f}")

GNN  → Recall@10: 0.0354 | NDCG@10: 0.0278
POP  → Recall@10: 0.0182 | NDCG@10: 0.0134


In [13]:
EXPORT_DIR = '../Data Folder'
os.makedirs(EXPORT_DIR, exist_ok=True)

model.eval()
with torch.no_grad():
    clip_f = torch.stack([item_dict[i][-CLIP_DIM:] for i in range(num_items)]).to(device)
    users_out, _ = model(edge_index, edge_weight, clip_f)
gnn_user_emb_np = users_out.cpu().numpy()
emb_path = os.path.join(EXPORT_DIR, 'gnn_user_embeddings.npy')
np.save(emb_path, gnn_user_emb_np)
print(f'GNN 유저 임베딩 저장 → {emb_path}')

metrics = {
    'recall_popularity': float(pop_recall), 'ndcg_popularity': float(pop_ndcg),
    'recall_gnn': float(gnn_recall), 'ndcg_gnn': float(gnn_ndcg),
    'eval_k': EVAL_K, 'num_users': int(num_users), 'num_items': int(num_items),
}
import json
metrics_path = os.path.join(EXPORT_DIR, 'gnn_metrics.json')
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print(f'메트릭 저장 → {metrics_path}')

comp_df = pd.DataFrame([
    {'model': 'Popularity Baseline', 'Recall@K': pop_recall, 'NDCG@K': pop_ndcg},
    {'model': 'MultiModal LightGCN', 'Recall@K': gnn_recall, 'NDCG@K': gnn_ndcg},
])
comp_df.to_csv(os.path.join(EXPORT_DIR, 'model_comparison_recall_ndcg.csv'), index=False, encoding='utf-8-sig')
print('model_comparison_recall_ndcg.csv 저장 완료')


GNN 유저 임베딩 저장 → ../Data Folder\gnn_user_embeddings.npy
메트릭 저장 → ../Data Folder\gnn_metrics.json
model_comparison_recall_ndcg.csv 저장 완료


In [ ]:
from sklearn.cluster import KMeans

n_clusters = 4
sample_size = min(10000, len(gnn_user_emb_np))
sample_embs = gnn_user_emb_np[:sample_size]
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(sample_embs)
if sample_size < len(gnn_user_emb_np):
    full_clusters = kmeans.predict(gnn_user_emb_np)
else:
    full_clusters = clusters
user_ids = user_encoder.inverse_transform(np.arange(len(user_encoder.classes_)))
cluster_df = pd.DataFrame({'customer_id': user_ids, 'cluster_label': full_clusters})
gnn_cluster_path = os.path.join(EXPORT_DIR, 'gnn_customer_clusters.csv')
cluster_df.to_csv(gnn_cluster_path, index=False, encoding='utf-8-sig')
print(f'GNN 군집 결과 저장 완료 → {gnn_cluster_path}')


GNN 군집 결과 저장 완료 → ../Data Folder\gnn_customer_clusters.csv


In [15]:
# 대시보드/Tableau 연동용 마스터 CSV (군집 + 거래 + 상품 메타)
tx_cols = ['customer_id', 'article_id', 't_dat']
if 'price_eur' in transactions.columns:
    tx_cols.append('price_eur')
elif 'price' in transactions.columns:
    tx_cols.append('price')
merged = transactions[[c for c in tx_cols if c in transactions.columns]].merge(cluster_df, on='customer_id', how='left')
art_cols = ['article_id', 'prod_name', 'product_group_name']
if 'colour_group_name' in articles.columns:
    art_cols.append('colour_group_name')
elif 'perceived_colour_master_name' in articles.columns:
    art_cols.append('perceived_colour_master_name')
merged = merged.merge(articles[art_cols].drop_duplicates('article_id'), on='article_id', how='left')
if 'price_eur' not in merged.columns and 'price' in merged.columns:
    merged['price_eur'] = merged['price'].astype(float)
elif 'price_eur' not in merged.columns:
    merged['price_eur'] = 0.0
if 'age' in customers.columns:
    merged = merged.merge(customers[['customer_id', 'age']].drop_duplicates('customer_id'), on='customer_id', how='left')
else:
    merged['age'] = np.nan
persona_map = {0: 'Cluster_0', 1: 'Cluster_1', 2: 'Cluster_2', 3: 'Cluster_3'}
merged['Cluster_Persona'] = merged['cluster_label'].map(persona_map)
cols_export = [c for c in ['t_dat', 'customer_id', 'article_id', 'price_eur', 'age', 'cluster_label', 'Cluster_Persona', 'prod_name', 'product_group_name', 'colour_group_name', 'perceived_colour_master_name'] if c in merged.columns]
dashboard_df = merged[cols_export].copy()
dashboard_df['t_dat'] = pd.to_datetime(dashboard_df['t_dat'])
dashboard_df.to_csv(os.path.join(EXPORT_DIR, 'Final_Dashboard_Master.csv'), index=False, encoding='utf-8-sig')
print(f'Final_Dashboard_Master.csv 저장 완료 → {EXPORT_DIR}')


Final_Dashboard_Master.csv 저장 완료 → ../Data Folder
